In [9]:
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
print("Key loaded:", api_key is not None and api_key.startswith("sk-"))

Key loaded: True


In [10]:
import csv

with open("data/opensanctions_sample.csv", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    rows = list(reader)

print(f"Loaded {len(rows)} entities")
print(rows[0].keys())   # see what columns we have
print(rows[0])          # inspect one row

Loaded 1308138 entities
dict_keys(['id', 'schema', 'name', 'aliases', 'birth_date', 'countries', 'addresses', 'identifiers', 'sanctions', 'phones', 'emails', 'program_ids', 'dataset', 'first_seen', 'last_seen', 'last_change'])
{'id': 'NK-223CQDBzp8MRkdJMDiqXn3', 'schema': 'Company', 'name': 'Myanmar Yatai International Holding Group Co., LTD.', 'aliases': 'Myanmar Yatai International Holding Group Co., Ltd;SHWE KOKKO SPECIAL ECONOMIC ZONE;Shwe Kokko Special Economic Zone;YATAI NEW CITY;YATAI SMART INDUSTRIAL NEW CITY;Yatai New City;Yatai Smart Industrial New City', 'birth_date': '', 'countries': 'mm', 'addresses': 'HPA-AN CITY;Hpa-An City;Shwe Kokko Village, Myawaddy Township, Karen State', 'identifiers': '103919088;GQBPAV1TFF41;PW2USM6LNCJ9;PW2XZT68KVW9;PW3LMJ5YB3M3', 'sanctions': 'GLOMAG - Executive Order 13818 (Global Magnitsky);Reciprocal - Active - 2025-09-08', 'phones': '', 'emails': '', 'program_ids': 'US-GLOMAG', 'dataset': 'OpenCorporates;US OFAC Press Releases;US OFAC Special

In [11]:
# Keep only rows that actually have sanctions info (richer for KYCC narratives)
# and are either a Person or a Company (the two schemas most relevant to KYCC/AML)
filtered = [
    r for r in rows
    if r["schema"] in ("Person", "Company")
    and r["sanctions"].strip() != ""
]

print(f"Filtered down to {len(filtered)} sanctioned entities")

# Take a sample so we're not embedding hundreds of thousands of docs
import random
random.seed(42)
sample = random.sample(filtered, min(300, len(filtered)))

print(f"Working sample: {len(sample)} entities")
print(sample[0])

Filtered down to 248167 sanctioned entities
Working sample: 300 entities
{'id': 'us-fed-excl-jose-claudio-plaza-33023-miramar', 'schema': 'Person', 'name': 'JOSE CLAUDIO PLAZA', 'aliases': '', 'birth_date': '1961-10-27', 'countries': 'us', 'addresses': '7949 ALHAMBRA BLVD, MIRAMAR, FL 33023', 'identifiers': '', 'sanctions': '1128b4 - 2003-12-18', 'phones': '', 'emails': '', 'program_ids': 'US-HHS-OIG', 'dataset': 'US Health and Human Services Inspector General Exclusions', 'first_seen': '2024-05-08T21:57:02', 'last_seen': '2026-07-18T22:38:09', 'last_change': '2026-01-06T21:57:01'}


In [13]:
def row_to_narrative(row: dict) -> str:
    """Convert one OpenSanctions row into a short KYCC-style profile narrative."""
    
    name = row["name"]
    entity_type = row["schema"]
    country = row["countries"] or "unknown country"
    dob = row["birth_date"] or "unknown date of birth"
    address = row["addresses"] or "no address on file"
    program = row["program_ids"] or "unspecified program"
    dataset = row["dataset"]
    sanctions_raw = row["sanctions"]

    if entity_type == "Person":
        subject_line = f"{name} is an individual associated with {country}."
        dob_line = f"Recorded date of birth: {dob}."
    else:
        subject_line = f"{name} is a corporate entity registered in or associated with {country}."
        dob_line = ""

    narrative = f"""
{subject_line} {dob_line}
Address on file: {address}.
This entity appears on the following watchlist: {dataset}, under program identifier(s): {program}.
Sanctions record: {sanctions_raw}.
Source: OpenSanctions consolidated database. Entity ID: {row['id']}.
""".strip()

    return narrative


# Build the full set of documents
documents = [row_to_narrative(r) for r in sample]

# Sanity check — read a couple
print(documents[0])
print("\n---\n")
print(documents[1])

JOSE CLAUDIO PLAZA is an individual associated with us. Recorded date of birth: 1961-10-27.
Address on file: 7949 ALHAMBRA BLVD, MIRAMAR, FL 33023.
This entity appears on the following watchlist: US Health and Human Services Inspector General Exclusions, under program identifier(s): US-HHS-OIG.
Sanctions record: 1128b4 - 2003-12-18.
Source: OpenSanctions consolidated database. Entity ID: us-fed-excl-jose-claudio-plaza-33023-miramar.

---

Скрибикін Анатолій Миколайович is an individual associated with ru. Recorded date of birth: 1960-01-13.
Address on file: no address on file.
This entity appears on the following watchlist: Ukraine NSDC State Register of Sanctions, under program identifier(s): UA-SA1644.
Sanctions record: 726/2022 - active - 2022-10-19 - 2027-10-19.
Source: OpenSanctions consolidated database. Entity ID: NK-KbduHTpHVNcX8FUjzHE5bK.


In [14]:
import importlib
import src.chunking
importlib.reload(src.chunking)
from src.chunking import chunk_text

long_chunks = chunk_text(long_text, chunk_size=500, overlap_sentences=1)
print(f"Long text ({len(long_text)} chars) produced {len(long_chunks)} chunks")
for i, c in enumerate(long_chunks):
    print(f"\n--- Chunk {i} ({len(c)} chars) ---")
    print(c[:150], "...")

Long text (2849 chars) produced 4 chunks

--- Chunk 0 (853 chars) ---
JOSE CLAUDIO PLAZA is an individual associated with us. Recorded date of birth: 1961-10-27. Address on file: 7949 ALHAMBRA BLVD, MIRAMAR, FL 33023. Th ...

--- Chunk 1 (800 chars) ---
Entity ID: NK-KbduHTpHVNcX8FUjzHE5bK. Margaret Lotz is an individual associated with us. Recorded date of birth: 1947-07-31. Address on file: 1309 N A ...

--- Chunk 2 (497 chars) ---
Address on file: 3 OLD COUNTY ROAD, VEAZIE, ME 04401;VEAZIE, ME 04401. This entity appears on the following watchlist: US Health and Human Services In ...

--- Chunk 3 (881 chars) ---
East Star Company is a corporate entity registered in or associated with ir. Address on file: Setareh Shargh Company Building, Mooalem Road, 4th Stree ...


In [16]:
# Apply to all documents: for now, each stays as 1 chunk, but the code is ready to scale
all_chunks = []
chunk_metadata = []  # keep track of which entity each chunk came from

for doc, row in zip(documents, sample):
    doc_chunks = chunk_text(doc, chunk_size=500, overlap_sentences=1)
    for chunk in doc_chunks:
        all_chunks.append(chunk)
        chunk_metadata.append({"entity_id": row["id"], "name": row["name"]})

print(f"Total chunks: {len(all_chunks)}")

Total chunks: 658


In [17]:
# Get embeddings from OpenAI
# -> turn each chunk into a vector

from openai import OpenAI

client = OpenAI() # Reads OPENAI_API_KEY from env automatically

def get_embedding(text: str, model: str = "text-embedding-3-small") -> list[float]:
    response = client.embeddings.create(input=text, model=model)
    return response.data[0].embedding

# Test on one chunk
test_vec = get_embedding(all_chunks[0])
print(f"Embedding length: {len(test_vec)}")
print(f"First 5 values: {test_vec[:5]}")

Embedding length: 1536
First 5 values: [-0.01397705078125, 0.00453948974609375, 0.0179290771484375, 0.06268310546875, -0.022735595703125]


In [19]:
# Now embed all chunks
import json
import os

EMBEDDINGS_CACHE_PATH = "data/embeddings_cache.json"

def get_embeddings_batch(texts: list[str], model: str = "text-embedding-3-small", batch_size: int = 100) -> list[list[float]]:
    """
    Embed a list of texts in batches bc API has per-request limit
    """
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        response = client.embeddings.create(input=batch, model=model)
        # response.data is returned in the same order as the input batch
        batch_embeddings = [item.embedding for item in response.data]
        all_embeddings.extend(batch_embeddings)
        print(f"Embedded {i + len(batch)}/{len(texts)}")

    return all_embeddings

if os.path.exists(EMBEDDINGS_CACHE_PATH):
    print("Loading cached embeddings...")
    with open(EMBEDDINGS_CACHE_PATH, "r") as f:
        chunk_embeddings = json.load(f)
else:
    print("No cache found, calling API...")
    chunk_embeddings = get_embeddings_batch(all_chunks)
    with open(EMBEDDINGS_CACHE_PATH, "w") as f:
        json.dump(chunk_embeddings, f)
    print("Cached to disk.")

print(f"Total embeddings: {len(chunk_embeddings)}")



Loading cached embeddings...
Total embeddings: 658


In [20]:
import numpy as np

# Convert to numpy arrays once -> so we're not re-converting on every search
chunk_embeddings_np = np.array(chunk_embeddings)  # shape: (658, 1536)

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """
    Measures cosine similarity btwn 2 vectors (ignoring magnitude)
    a: single query vector, shape (1536, )
    b: matrix of chunk vectors, shape (N, 1536)
    output: returns arr of N similarity scores (1/chunk)
    """
    a_norm = a / np.linalg.norm(a)
    b_norm = b / np.linalg.norm(b, axis=1, keepdims=True)
    return b_norm @ a_norm # dot prod of normalized vecs = cosine similarity

def search(query: str, top_k: int=5):
    query_vec = get_embedding(query)
    scores = cosine_similarity(query_vec, chunk_embeddings_np)

    # get indices of top k highest scores
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []
    for idx in top_indices:
        results.append({
            "score": float(scores[idx]),
            "chunk": all_chunks[idx],
            "entity_id": chunk_metadata[idx]["entity_id"],
            "name": chunk_metadata[idx]["name"],
        })
    return results

# Test
results = search("individuals sanctioned in relation to Iran missile proliferation")
for r in results:
    print(f"\nScore: {r['score']:.4f} | {r['name']}")
    print(r['chunk'][:200], "...")


Score: 0.5520 | Hanista Programing Group
Hanista Programing Group is a corporate entity registered in or associated with ir. Address on file: no address on file. This entity appears on the following watchlist: US OFAC Press Releases;US OFAC  ...

Score: 0.5403 | Ali Saddam Hussein Al-Tikriti
This entity appears on the following watchlist: Australian Sanctions Consolidated List;Belgian Financial Sanctions;EU Financial Sanctions Files (FSF);French National Asset Freezing System;Japan Econom ...

Score: 0.5162 | Ali Saddam Hussein Al-Tikriti
Sanctions record: ;(CE) 924/2004 du 29/04/2004 (ONU Irak - RCSNU 1518 (2003) et R (CE) 1210/2003) - fils de Samira Shahbandar et de Saddam Hussein;1518 (Iraq) - 2025-12-11;924/2004 (OJ L163);IRAQ2 - U ...

Score: 0.5132 | Iran Mobin Electronic Development Company
Iran Mobin Electronic Development Company is a corporate entity registered in or associated with ir. Address on file: TEHRAN, AHMAD QASSIR BOKHAREST ST., ARGENTINE SQ.;Tehran, Ahmad Qassir B